# Transformers

In [1]:
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Project Description

In this project I will go over transformer language models. Specifically I will be explaining the concepts introduced in the paper [Attention Is All You Need](https://arxiv.org/pdf/1706.03762). This is a very important paper that not only changed the field of computer science but also non technical fields. The notebook will cover the architecture which includes encoder, decoder, attention, etc. I will also be covering the math behind it as well as training. I have also implemented my own transformer using Jax. The code for that can be found in the `./src`and `main.py`. Lets get into it! 

## Background

One of the many advantages of transformer models is that they can process large amounts of text data at once. Previously Long Short Term Memory (LSTM) networks/recurrent networks were used to model language but one of their drawbacks is that they processed data one word at a time. This made it difficult for models to  understand text from the beginning of a sequence if the sequence was large, suffered from vanishing gradients, and processed data one token at a time which is computational expensive for large data sets. With the transformer architecture models are capable of better understanding data throughout the sequence and process large amounts of data at once. The specific task discussed in the paper was language translation. Which we will also try to implement. Usually for tasks like text generation such as the more common GPT models, only the decoder part of the model is used. Because I want to understand the transformer model thoroughly I want to implement the encoder and decoder.

## Model Architecture

The transformer architecture from the paper Attention Is All You Need consists of a N decoder and encoder blocks. The encoder block consits of 

#### Input Embeddings
The Encoder takes embeddings as input. Embeddings are vectors representations of a word/token. The embeddings are learnable parameters. In the original paper each token embedding is of size 512 defined as $d_{model}$. Here is an example of some text and what the tokens might look like.  

- **Sequence:**  lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod
tempor incididunt ut labore et dolore magna aliqua.
- **Tokens:**  (lorem ipsum dolor) (sit) (amet, consectetur) (adipiscing elit, sed do) (eiusmod) (tempor) (incididunt) (ut labore) (et dolore magna aliqua.)


- **Sequence:**  Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo
consequat.
- **Tokens:**  (Ut) (enim ad minim veniam,) (quis nostrud) (exercitation) (ullamco) (laboris) (nisi ut aliquip) (ex ea) (commodo consequat.)


Although tokens can be of any size, its easier to understand the attention mechanism if we only think of tokens as one word so in this notebook each token is just one word. Each token is also mapped to a id so that we so we can later look them up in our vocbulary. This is an example of what a token embedding might look like. 

- **Token:**  Dog
- **Token ID:** [12]
- **Embedding:** $\{e_1, e_2, \dots , e_{512} \}$


All together our input embedding is a matrix of size (seq_len, d_model). This is pretty straight forward. It just means we have a sequence of tokens of size seq_len and each is represented by a vector of size d_model. Because we focused on the transformer we will not be going into how we got these tokens. For now we will just assume we have them. 

### Encoder

Lets get into the encoder. The encoder part of the model is made up of "N = 6 identical layers". Essentially we repeat the encoder 6 times. Each encder block contains 3 layers of normalization followed by attention blocks. The idea is that ...

After getting our input embedding we then apply positional encoding. We use $\sin$ and $\cos$ functions to apply to the token vector and add it.

Here is an example.


Positional encoding helps us track the location of the words and their importance there. This is calculated once. 

We then pass the embeddings into the multi head attention and also to a add and norm block. Lets go over the multi head attention mechanism. 

### Decoder
The decoder will have

#### Attention
Attention is defined as.  
$$ Attention(Q, K, V) = softmax(\frac{QK^{T}}{\sqrt{d_k}})V $$

This equation may look very confusing so lets break it down part by part. 

We have a input embeddings of size (${seq_len}$, $d_{model}$). For a basic visual example lets say "The dog ate" is our input. Then our input embedding will look something like this. For the purpose of this example ${seq_len}=5$, $d_{model}=10$, and $d_k=10$ 

This would give us a matrix that looks something like this. 
$$
Q_{5,10} = \begin{bmatrix}
8 & 63 & 39 & 36 & 81 & 3 & 73 & 8 & 50 & 2 \\
61 & 90 & 99 & 92 & 42 & 25 &  30 & 5 & 4&  79 \\
11 &  99 & 10 & 34 & 72 & 9 & 74 & 84 & 65 & 62 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 
\end{bmatrix}
$$
$$
K^{T}_{10,5} = \begin{bmatrix}
8 & 63 & 39 & 36 & 81 & 3 & 73 & 8 & 50 & 2 \\
10 & 2 & 2 & 2 & 2 & 2 & 2 & 2 & 2 & 81 \\
1 & 2 & 2 & 2 & 2 & 2 & 2 & 2 & 2 & 10 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
8 & 63 & 39 & 36 & 81 & 3 & 73 & 8 & 50 & 2 \\
10 & 2 & 2 & 2 & 2 & 2 & 2 & 2 & 2 & 81 \\
1 & 2 & 2 & 2 & 2 & 2 & 2 & 2 & 2 & 10 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 
\end{bmatrix}
$$
$$
V_{5,10} = \begin{bmatrix}
8 & 63 & 39 & 36 & 81 & 3 & 73 & 8 & 50 & 2 \\
61 & 90 & 99 & 92 & 42 & 25 &  30 & 5 & 4&  79 \\
11 &  99 & 10 & 34 & 72 & 9 & 74 & 84 & 65 & 62 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 
\end{bmatrix}
$$

$$
QK^{T}_{5,10} = \begin{bmatrix}
8 & 63 & 39 & 36 & 81 & 3 & 73 & 8 & 50 & 2 \\
61 & 90 & 99 & 92 & 42 & 25 &  30 & 5 & 4&  79 \\
11 &  99 & 10 & 34 & 72 & 9 & 74 & 84 & 65 & 62 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 
\end{bmatrix}
$$
$$
\frac{QK^{T}}{\sqrt{d_k}} = \begin{bmatrix}
8 & 63 & 39 & 36 & 81 & 3 & 73 & 8 & 50 & 2 \\
61 & 90 & 99 & 92 & 42 & 25 &  30 & 5 & 4&  79 \\
11 &  99 & 10 & 34 & 72 & 9 & 74 & 84 & 65 & 62 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 
\end{bmatrix}
$$
$$
softmax(\frac{QK^{T}}{\sqrt{d_k}}) = \begin{bmatrix}
8 & 63 & 39 & 36 & 81 & 3 & 73 & 8 & 50 & 2 \\
61 & 90 & 99 & 92 & 42 & 25 &  30 & 5 & 4&  79 \\
11 &  99 & 10 & 34 & 72 & 9 & 74 & 84 & 65 & 62 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 
\end{bmatrix}
$$
$$
Attntn_{5,10} = softmax(\frac{QK^{T}}{\sqrt{d_k}}) V  = \begin{bmatrix}
8 & 63 & 39 & 36 & 81 & 3 & 73 & 8 & 50 & 2 \\
61 & 90 & 99 & 92 & 42 & 25 &  30 & 5 & 4&  79 \\
11 &  99 & 10 & 34 & 72 & 9 & 74 & 84 & 65 & 62 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 
\end{bmatrix}
$$

Now that was attention which is where we process the matrix all at once. 

#### Multi Head Attention
Multi head attention is very very similar but instead of calculating attention all at once we split the matrix in `h` heads. And we calcualte attention for each head. This way each head contains a small part of each query key and value and each head can .... The once we have the attention scores computed for each of the heads we combine them back to get the attention matrix. 

So our matrix from above was 
$$
Q_{5,10} = \begin{bmatrix}
8 & 63 & 39 & 36 & 81 & 3 & 73 & 8 & 50 & 2 \\
61 & 90 & 99 & 92 & 42 & 25 &  30 & 5 & 4&  79 \\
11 &  99 & 10 & 34 & 72 & 9 & 74 & 84 & 65 & 62 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 
\end{bmatrix}
$$
to do multi head attention with 5 heads we will have. 
$$
Q_1 = \begin{bmatrix}
8 & 63 \\
61 & 90  \\
11 &  99 \\
0 & 0   \\
0 & 0 
\end{bmatrix}
Q_2 = \begin{bmatrix}
 39 & 36  \\
 99 & 92  \\
 10 & 34  \\
 0 & 0  \\
 0 & 0 
\end{bmatrix}
Q_3 = \begin{bmatrix} 81 & 3 \\ 42 & 25 \\ 72 & 9 \\ 0 & 0 \\ 0 & 0 \end{bmatrix}
Q_4 = \begin{bmatrix} 73 & 8 \\ 30 & 5 \\ 74 & 84 \\ 0 & 0 \\ 0 & 0 \end{bmatrix}
Q_5 = \begin{bmatrix} 50 & 2 \\ 4 & 79 \\ 65 & 62 \\ 0 & 0 \\ 0 & 0 \end{bmatrix}
$$
We would do the same for $K^T$ and for $V$

So our equation would become 

$ \forall i \in \text{h heads} \hspace{10px} head_i =  AttentionHead(Q_i,K_i, V_i) = softmax(\frac{Q_iK^{T}_i}{\sqrt{d_k}})V_i $ 

$  Attention(Q,K, V) = concat_{n=1}^{\text{h heads}}(head_i)W^{O}$ 

## Training 
So the goal is to train a model that learns Spansih to English and Nahuatl. To do this I got some insipration from this. 
Because Nahuatl is a low resource language ( samples) the plan is to train the model on the Spanish to English dataset which is very large. This way in a second phase of training the encoder has already learned patterns of spanish and can focus on learning the patterns of Nahuatl in the decoder. 

### Phase 1
So for phase 1 we have our Spanish to English dataset which is at about n samples. Below are the hyper parameters I choose for the model.

```
BATCH_SIZE=32,
EPOCHS=50,
LR=3e-4,
SEQ_LEN=128,
D_MODEL=512,
D_FF=2048,
H=8,
N=6,
DROPOUT_SCHEDULE={0: 0.15, 15: 0.25, 30: 0.3},
WEIGHT_DECAY=0.05

```

Initially the model was overfitting by a lot so I chose a dropout rate schedule. The idea is that the first few iterations have minimal dropout so the model can learn as much as possible quickly. At around iteration 15 it starts to overfit so we introduce a higher level of dropout so that the model can truly learn. To further prevent overfitting at around iteration 30 we update the dropout to be higher. This way we ensure that the model learns and does not overfit in the final iterations. The epochs was set about 100 but I really only ran it 50. I continued the training and got a better eval loss at around 70 but it really is not much improvement. This took like about 12+ hours if i remember correctly. 

## Phase 2
For phase two we had a lot less data. The dataset is of size n but on top of this I added n Spanish to English samples. So our phase 2 training included Spanish to English and Spanish to Nahuatl samples. 

When training we used the best checkpoint from phase 1 which was `70`. 

## Results 

## Conclusion